# B06 — Stage 1: rho* scout on the Burgold pilot

Reads the versioned scout artifacts (CSVs) that live in this Mantis workspace.

Locked estimand per EPISTASIS_ID_LOCK: `delta = m_ab - m_a0 - m_0b + mu` on `FC_500x`.
Sign identified iff `|delta| > rho*(|d_a| + |d_b|)`.

In [ ]:
import pandas as pd, numpy as np, json, pathlib

base = pathlib.Path('.')  # workspace root in Mantis
ct = pd.read_csv('results/bounds/guide_pair_contrasts.csv')
calls = pd.read_csv('results/bounds/gene_pair_calls.csv')
summary = json.load(open('results/audit/audit_summary.json'))
ct.head()

In [ ]:
# 1) Robustness radius distribution (Sec 5.4 / 24.1)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(ct['rho_star'], bins=30, log=True)
ax[0].axvline(0.10, color='r', ls='--', label=r'$\rho=0.10$')
ax[0].set_xlabel(r'$\rho^* = |\delta|/(|d_a|+|d_b|)$'); ax[0].set_ylabel('guide pairs (log)')
ax[0].legend()

# 2) Identified fraction vs drift envelope (monotone)
resolved = calls[calls.gene_sign != 'UNRESOLVED'].groupby('rho')['gene_sign'].count()
total = calls.groupby('rho')['gene_sign'].count()
frac = (resolved / total * 100)
ax[1].plot(frac.index, frac.values, 'o-')
ax[1].set_ylabel('% gene pairs identified'); ax[1].set_xlabel(r'context-drift $\rho$'); ax[1].set_ylim(0, 100)
fig.suptitle('B06 scout: perturbation-strength robustness (pilot library)')
plt.tight_layout(); plt.show()

In [ ]:
# 3) The guardrail (Sec 4.1 / Fig 2): delta = s_a*s_b*gamma, sign invariant
rng = np.random.default_rng(0)
sa, sb = rng.uniform(0.2, 3, 500), rng.uniform(0.2, 3, 500)
gamma = rng.normal(0, 1, 500)
delta = sa*sb*gamma
print('sign concordance (stable efficacy):', np.mean(np.sign(delta) == np.sign(gamma)))

In [ ]:
# 4) Robust interval vs rho for the known paralog hit PRMT1-PRMT5
row = ct[(ct.gene_a=='PRMT1') & (ct.gene_b=='PRMT5')].iloc[0]
rhos = np.linspace(0, 0.5, 51)
half = rhos*(abs(row.m_a0-row.mu)+abs(row.m_0b-row.mu))
fig, ax = plt.subplots(figsize=(6,3.5))
ax.plot(rhos, row.delta-half, 'r', label='lower'); ax.plot(rhos, row.delta+half, 'b', label='upper')
ax.axhline(0, color='k', lw=.5); ax.axvline(row.rho_star, color='g', ls='--', label=f"$\\rho^*$={row.rho_star:.2f}")
ax.set_xlabel(r'$\rho$'); ax.set_title('PRMT1-PRMT5 identified set'); ax.legend()
plt.tight_layout(); plt.show()

## Reading the output
- The rho* histogram shows how close each guide pair sits to the sign boundary.
- The identified fraction decays monotonically with rho (0.10 = 17/25?).
- Guardrail confirms stable efficacy cannot flip signs (concordance = 1.0).

Next: guide-identity holdout (split guides, refit rho* band, test held-out guide-pair phenotype).